# P5 정기결제 탐지 분석

## 분석 목적
거래내역에서 반복 결제 패턴을 탐지하여 고객별 구독 및 고정비 목록을 추출한다.

## 탐지 기준
- 승인된 거래만 분석
- merchant_name이 존재하는 거래
- 동일 고객·동일 가맹점에서 3개월 이상 반복 결제
- 동일 금액 반복
- 결제일 차이 ±3일 범위

## 검증 방법
실제 SUBSCRIPTION 카테고리와 탐지 결과를 비교하여 탐지 성능을 확인한다.

In [2]:
import pandas as pd

df = pd.read_csv("../data/transactions.csv")

print("전체 거래 건수:", len(df))
print("고객 수:", df["customer_id"].nunique())
print("분석 기간:", df["transaction_datetime"].min(), "~", df["transaction_datetime"].max())

df.head()

전체 거래 건수: 318145
고객 수: 1000
분석 기간: 2025-01-01 00:00:37 ~ 2025-12-31 23:59:36


,transaction_id,customer_id,persona_id,age_group,gender,occupation,annual_income,household_type,transaction_datetime,merchant_name,category,amount,payment_method,card_product,transaction_status
0,TX000001,C0487,P04,30s,F,OFFICE_WORKER,34600000,SINGLE,2025-01-01 00:00:37,쿠팡와우,SUBSCRIPTION,7890,CHECK_CARD,CARD_A,APPROVED
1,TX000002,C0550,P16,30s,F,OFFICE_WORKER,77680000,NEWLYWED,2025-01-01 00:03:22,티빙,SUBSCRIPTION,13900,CHECK_CARD,CARD_C,APPROVED
2,TX000003,C0735,P08,30s,M,OFFICE_WORKER,55440000,NEWLYWED,2025-01-01 00:14:24,어도비 Creative Cloud,SUBSCRIPTION,30800,CHECK_CARD,CARD_A,APPROVED
3,TX000004,C0171,P01,30s,M,OFFICE_WORKER,43300000,SINGLE,2025-01-01 00:25:19,스팀,ENTERTAINMENT,30300,CHECK_CARD,CARD_A,APPROVED
4,TX000005,C0739,P03,40s,M,OFFICE_WORKER,69610000,SINGLE_INCOME,2025-01-01 00:33:05,CU,CONVENIENCE,7100,CREDIT_CARD,CARD_D,APPROVED


In [3]:
import sys
sys.path.append("../src")

from recurring import detect_recurring

result = detect_recurring(df)

print("정기결제 탐지 후보:", len(result))
result.head(10)

정기결제 탐지 후보: 2664


,customer_id,merchant_name,active_months,payment_count,avg_amount,min_amount,max_amount,first_date,last_date,min_day,max_day,day_range
3,C0001,LG유플러스,11,11,64200.0,64200,64200,2025-01-07 13:18:27,2025-12-07 11:02:19,7,7,0
66,C0002,SK텔레콤,12,12,31000.0,31000,31000,2025-01-19 15:21:24,2025-12-19 11:34:12,19,19,0
118,C0002,쿠팡와우,12,12,7890.0,7890,7890,2025-01-27 06:18:39,2025-12-27 06:06:45,26,28,2
132,C0003,LG유플러스,12,12,29700.0,29700,29700,2025-01-21 11:16:10,2025-12-21 11:19:18,21,21,0
143,C0003,넷플릭스,12,12,17000.0,17000,17000,2025-01-05 01:13:28,2025-12-06 06:53:29,4,6,2
155,C0003,밀리의서재,12,12,9900.0,9900,9900,2025-01-23 01:25:36,2025-12-22 04:35:43,21,23,2
187,C0003,쿠팡와우,12,12,7890.0,7890,7890,2025-01-18 06:37:39,2025-12-19 05:02:13,18,19,1
192,C0003,티빙,12,12,13900.0,13900,13900,2025-01-03 02:09:54,2025-12-02 00:38:22,2,3,1
206,C0004,LG유플러스,12,12,30700.0,30700,30700,2025-01-22 14:41:39,2025-12-22 12:07:29,22,22,0
230,C0004,밀리의서재,12,12,9900.0,9900,9900,2025-01-18 04:17:01,2025-12-17 05:45:17,16,18,2


In [4]:
# 탐지 결과에 실제 카테고리 연결
result_category = result.merge(
    df[["customer_id", "merchant_name", "category"]].drop_duplicates(),
    on=["customer_id", "merchant_name"],
    how="left"
)

# 카테고리 표기 통일
result_category["category_clean"] = (
    result_category["category"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# 고객-가맹점 기준 중복 제거
result_category = result_category.drop_duplicates(
    subset=["customer_id", "merchant_name"]
)

print("전체 탐지 후보:", len(result_category))
print()
print(result_category["category_clean"].value_counts())

전체 탐지 후보: 2664

category_clean
SUBSCRIPTION    1664
UTILITY         1000
Name: count, dtype: int64


In [5]:
# 실제 SUBSCRIPTION 데이터
subscription = df[
    df["category"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("SUBSCRIPTION")
].copy()

# merchant_name 결측은 반복 결제 여부를 판단할 수 없어 검증에서 제외
subscription_valid = subscription.dropna(subset=["merchant_name"])

actual_subscription = set(
    subscription_valid[
        ["customer_id", "merchant_name"]
    ].drop_duplicates().itertuples(index=False, name=None)
)

predicted_recurring = set(
    result[
        ["customer_id", "merchant_name"]
    ].drop_duplicates().itertuples(index=False, name=None)
)

true_positive = actual_subscription & predicted_recurring

recall = len(true_positive) / len(actual_subscription)
precision = len(true_positive) / len(predicted_recurring)

print("검증 가능한 실제 SUBSCRIPTION:", len(actual_subscription))
print("SUBSCRIPTION 탐지 성공:", len(true_positive))
print("전체 정기결제 탐지:", len(predicted_recurring))
print(f"Recall: {recall:.2%}")
print(f"SUBSCRIPTION 기준 Precision: {precision:.2%}")

검증 가능한 실제 SUBSCRIPTION: 1664
SUBSCRIPTION 탐지 성공: 1664
전체 정기결제 탐지: 2664
Recall: 100.00%
SUBSCRIPTION 기준 Precision: 62.46%


## 탐지 결과 및 검증

- 정기결제 탐지 후보: **2,664개**
- SUBSCRIPTION: **1,664개**
- UTILITY: **1,000개**
- merchant_name 결측을 제외한 실제 SUBSCRIPTION: **1,664개**
- SUBSCRIPTION 탐지 성공: **1,664개**
- Recall: **100.00%**
- SUBSCRIPTION 기준 Precision: **62.46%**

### 결과 해석
가맹점명이 존재하는 SUBSCRIPTION 거래는 모두 탐지하여 Recall 100%를 기록했다.

SUBSCRIPTION만 정답으로 정의할 경우 Precision은 62.46%였으나, 나머지 탐지 결과 1,000개는 일반 소비가 아니라 UTILITY 카테고리의 고정비였다. 따라서 SUBSCRIPTION 기준 Precision 수치는 통신비 등 정기적으로 발생하는 고정비까지 탐지한 결과를 오탐으로 처리하기 때문에 낮게 나타난다.

현재 탐지 규칙을 통해 구독 서비스와 정기성 고정비를 함께 식별할 수 있음을 확인했다.

In [6]:
from recurring import analyze

# 고객별 정기결제 분석 예시
example = analyze(df, "C0003", "2025-12")

example

{'customer_id': 'C0003',
 'month': '2025-12',
 'recurring_count': 5,
 'recurring_total': 78390,
 'recurring_list': [{'merchant_name': 'LG유플러스',
   'amount': 29700,
   'active_months': 12},
  {'merchant_name': '넷플릭스', 'amount': 17000, 'active_months': 12},
  {'merchant_name': '밀리의서재', 'amount': 9900, 'active_months': 12},
  {'merchant_name': '쿠팡와우', 'amount': 7890, 'active_months': 12},
  {'merchant_name': '티빙', 'amount': 13900, 'active_months': 12}]}